# DS317 – Lab 3 | Data Quality: <TÊN KHÍA CẠNH>
**Người phụ trách:** <tên>  |  **Khía cạnh:** <Completeness / Uniqueness / Validity / Consistency / Accuracy / Timeliness>

> Quy ước chung: điểm `score` ∈ [0,1] cho **từng bản ghi** (1 = tốt). Không sửa schema kết quả, không sửa các cell đánh dấu `# KHÔNG SỬA`.

## 0. Định nghĩa & luật (điền ngắn gọn)
| rule_id | Áp dụng cho bảng/cột | Mô tả luật | Cách tính điểm |
|---|---|---|---|
| R1 | | | |
| R2 | | | |

**Giả định / hạn chế:** (ví dụ: Accuracy là ước lượng bằng outlier, không có ground truth)

In [ ]:
# 1. Setup  (KHÔNG SỬA)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
# from google.colab import drive; drive.mount('/content/drive')

DIMENSION = "completeness"   # TODO: đổi đúng khía cạnh của bạn (chữ thường)
SCHEMA = ["record_id","table","version","batch","dimension","score","is_violation","rule_id","reason"]

In [ ]:
# 2. Cấu hình chung – cả nhóm dùng CÙNG nội dung cell này (KHÔNG SỬA)
CFG = {
  "paths": {                      # paths[version][table]
    "v0": {"sales": "...", "product": "...", "inventory": "..."},   # raw
    "v1": {"sales": "...", "product": "...", "inventory": "..."},   # đã làm sạch
    "v2": {"sales": "...", "product": "...", "inventory": "..."},   # modeling dataset
  },
  "keys":     {"sales": [], "product": [], "inventory": []},        # cột khóa; [] nếu không có
  "date_col": {"sales": None, "product": None, "inventory": None},  # None nếu bảng không có ngày
  "batch_freq": "W",   # "W" = tuần, "M" = tháng
  "split_dates": {"train_end": "...", "valid_end": "..."},          # mốc chia Train/Valid/Test theo thời gian
}

In [ ]:
# 3. Hàm hỗ trợ  (KHÔNG SỬA)
def load_table(table, version):
    return pd.read_csv(CFG["paths"][version][table])

def make_batch(df, table):
    col = CFG["date_col"][table]
    if col is None or col not in df.columns:
        return pd.Series("NA", index=df.index)
    d = pd.to_datetime(df[col], errors="coerce")
    return d.dt.to_period(CFG["batch_freq"]).astype(str).where(d.notna(), "NaT")

def make_record_id(df, table):
    keys = CFG["keys"][table]
    if keys:
        return df[keys].astype(str).agg("|".join, axis=1) + "#" + df.groupby(keys).cumcount().astype(str)
    return pd.Series(table + "_" + df.index.astype(str), index=df.index)  # dùng index gốc khi không có khóa

## 4. Hàm tính điểm theo bản ghi  ← PHẦN CỦA BẠN (`TODO`)
Trả về DataFrame **cùng index với `df`** gồm 4 cột: `score`, `is_violation`, `rule_id`, `reason`. Nếu một bản ghi vi phạm nhiều luật, ghi `rule_id` dạng `R1;R3` và `reason` tương ứng.

In [ ]:
# TODO: viết logic của khía cạnh bạn. Ví dụ mẫu (Completeness) – xoá/thay bằng logic của bạn:
def score_table(df, table):
    required = [c for c in df.columns]            # TODO: chọn cột bắt buộc theo từng bảng
    empty = df[required].isna() | (df[required].astype(str).apply(lambda s: s.str.strip()).isin(["", "N/A", "nan", "None"]))
    score = 1 - empty.mean(axis=1)
    out = pd.DataFrame(index=df.index)
    out["score"] = score
    out["is_violation"] = score < 1
    out["rule_id"] = np.where(out["is_violation"], "R1", "")
    out["reason"] = empty.apply(lambda r: ",".join(r.index[r]), axis=1)
    return out

In [ ]:
# 5. Chạy cho mọi bảng × phiên bản  (KHÔNG SỬA)
parts = []
for version, tables in CFG["paths"].items():
    for table in tables:
        try:
            df = load_table(table, version)
        except Exception as e:
            print(f"[skip] {version}/{table}: {e}"); continue
        res = score_table(df, table)
        assert res.index.equals(df.index), "score_table phải trả về cùng index với df"
        res = res.assign(record_id=make_record_id(df, table), table=table, version=version,
                         batch=make_batch(df, table), dimension=DIMENSION)
        parts.append(res[SCHEMA])
result = pd.concat(parts, ignore_index=True)
assert result["score"].between(0, 1).all(), "score phải nằm trong [0,1]"
assert list(result.columns) == SCHEMA
result.head()

In [ ]:
# 6. Lưu kết quả  (KHÔNG SỬA) – nộp file này
result.to_parquet(f"{DIMENSION}.parquet", index=False)
print(len(result), "bản ghi ->", f"{DIMENSION}.parquet")

In [ ]:
# 7. Tổng hợp theo bảng × phiên bản  (KHÔNG SỬA)
by_tv = (result.groupby(["table","version"])
               .agg(score_mean=("score","mean"), violation_rate=("is_violation","mean"), n=("score","size"))
               .round(4))
by_tv

In [ ]:
# 8. Tổng hợp theo lô/giai đoạn + biểu đồ  (KHÔNG SỬA)
by_tvb = result.groupby(["table","version","batch"])["score"].mean().reset_index()
by_tvb = by_tvb[~by_tvb["batch"].isin(["NA","NaT"])]

tables_with_time = by_tvb["table"].unique()
if len(tables_with_time) == 0:
    print("Không có bảng nào có trục thời gian -> bỏ qua biểu đồ theo lô.")
for t in tables_with_time:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for v, g in by_tvb[by_tvb.table == t].groupby("version"):
        ax.plot(g["batch"], g["score"], marker="o", ms=3, label=v)
    ax.set_title(f"{DIMENSION} theo lô – bảng {t}"); ax.set_ylim(0, 1.02)
    ax.set_xlabel("batch"); ax.set_ylabel("score trung bình"); ax.legend()
    step = max(1, len(g) // 12); ax.set_xticks(ax.get_xticks()[::step]); plt.xticks(rotation=45)
    plt.tight_layout(); plt.savefig(f"{DIMENSION}_{t}_by_batch.png", dpi=150); plt.show()

# bar so sánh phiên bản theo bảng
piv = by_tv["score_mean"].unstack("version")
piv.plot(kind="bar", figsize=(6, 3.5), ylim=(0, 1.02), title=f"{DIMENSION}: bảng × phiên bản")
plt.tight_layout(); plt.savefig(f"{DIMENSION}_by_table_version.png", dpi=150); plt.show()

## 9. Nhận xét ngắn (3–5 dòng)
- Bảng/phiên bản nào điểm thấp nhất, vì sao?
- v1 so với v0: bước làm sạch có cải thiện khía cạnh này không?
- Giai đoạn (batch) nào bất thường?

**File nộp:** `<dimension>.parquet`, các ảnh `*.png`, notebook này.